# 🧠 FASE 4: Pemodelan Deep Learning (LSTM)
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini melatih model Deep Learning **Long Short-Term Memory (LSTM)** menggunakan framework Keras/TensorFlow.
LSTM dirancang khusus untuk memecahkan masalah prediktif *time-series* karena kemampuannya "mengingat" pola masa lalu (dalam hal ini, 24 jam ke belakang).

**Target kita:** Mengalahkan skor RMSE dari Random Forest (7.69) di Fase 3.


## 1. Import Library
Pastikan Anda sudah menginstall `tensorflow` atau `keras`. Jika belum, jalankan `pip install tensorflow` di terminal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

# Evaluasi
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow / Keras untuk LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Data Processed & Scaler


In [ ]:
DATA_DIR = r'D:\Kuliah Praktik\KP BRIN\data\processed'

train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
test_scaled = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

scaler_y = joblib.load(os.path.join(DATA_DIR, 'scaler_y.pkl'))

# Target
target_col = 'pm25'
print(f"Bentuk awal Train 2D : {train_scaled.shape}")
print(f"Bentuk awal Test 2D  : {test_scaled.shape}")


## 3. Transformasi Data: Tabel 2D menjadi Tensor 3D (Sliding Window)
Algoritma konvensional (seperti XGBoost) memproses data sebaris demi sebaris (2D). 
Namun, LSTM membaca data berupa balok (*Tensor 3D*) dengan format: `[Jumlah Sampel, Jendela Waktu, Jumlah Fitur]`.

Kita menggunakan parameter `TIME_STEPS = 24`, yang artinya untuk menebak PM2.5 di jam ke-25, model diberi "contekan" cuaca dan polusi selama 24 jam sebelumnya.


In [ ]:
def create_sequences(df, target_col, time_steps=24):
    X, y = [], []
    data_array = df.values
    target_idx = df.columns.get_loc(target_col)
    
    for i in range(len(data_array) - time_steps):
        # Ambil blok 24 jam ke belakang sebagai Input (X)
        X.append(data_array[i:(i + time_steps)])
        # Ambil jam ke-25 (sekarang) sebagai Target (y)
        y.append(data_array[i + time_steps, target_idx])
        
    return np.array(X), np.array(y)

TIME_STEPS = 24

# Transformasi Data Train
X_train_seq, y_train_seq = create_sequences(train_scaled, target_col, TIME_STEPS)

# Transformasi Data Test
X_test_seq, y_test_seq = create_sequences(test_scaled, target_col, TIME_STEPS)

print(f"Bentuk Tensor X_train : {X_train_seq.shape} --> [Sampel, TimeSteps, Fitur]")
print(f"Bentuk Tensor y_train : {y_train_seq.shape}")
print(f"Bentuk Tensor X_test  : {X_test_seq.shape}")


## 4. Membangun Arsitektur Jaringan Saraf LSTM
Struktur jaringan kita:
1. **Input Layer:** Menerima ukuran `(24, 15)` -> 24 jam, 15 fitur (cuaca, waktu, lag).
2. **LSTM Layer 1 (64 unit):** Membaca urutan waktu, mengembalikan seluruh urutan (`return_sequences=True`).
3. **Dropout (0.2):** Mencegah *overfitting* dengan "mematikan" 20% saraf secara acak.
4. **LSTM Layer 2 (32 unit):** Mengekstrak kesimpulan akhir dari urutan waktu.
5. **Dense Output:** 1 Saraf tunggal (karena hanya memprediksi 1 nilai: PM2.5).


In [ ]:
# Membangun model berurutan (Sequential)
model = Sequential()

# Layer Input
model.add(Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])))

# Layer LSTM Pertama
model.add(LSTM(units=64, return_sequences=True))
model.add(Dropout(0.2)) # Mencegah overfitting

# Layer LSTM Kedua
model.add(LSTM(units=32, return_sequences=False))
model.add(Dropout(0.2))

# Layer Output (Dense)
model.add(Dense(units=1))

# Kompilasi Model (Optimizer Adam sangat cepat dan stabil)
model.compile(optimizer='adam', loss='mean_squared_error')

# Menampilkan struktur otak buatan kita
model.summary()


## 5. Melatih Model (Training)
Training Deep Learning dilakukan berulang-ulang (*epochs*).
Kita memasang pengaman `EarlyStopping`: Jika akurasi model pada data validasi tidak membaik setelah 10 *epoch*, *training* akan otomatis berhenti untuk menghemat waktu.


In [ ]:
print("Memulai pelatihan LSTM... (Akan memakan waktu 1-3 menit tergantung spesifikasi laptop)")

# Pengaman agar tidak Overfitting
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)

# Melatih Model
history = model.fit(
    X_train_seq, 
    y_train_seq,
    epochs=50,             # Maksimal 50 putaran
    batch_size=32,         # Membaca data per 32 baris agar memori tidak penuh
    validation_split=0.1,  # Menyisihkan 10% data latih untuk ujian mini (validasi)
    callbacks=[early_stop],
    verbose=1
)


### Kurva Pembelajaran (Learning Curve)
Grafik ini penting untuk melihat apakah LSTM belajar dengan baik. Garis biru (Training) dan oranye (Validation) idealnya harus turun bersama-sama.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Loss Latih (Train)')
plt.plot(history.history['val_loss'], label='Loss Ujian Mini (Validation)')
plt.title('Kurva Pembelajaran LSTM (Loss History)')
plt.xlabel('Epochs (Putaran Belajar)')
plt.ylabel('Mean Squared Error Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 6. Evaluasi Akhir & Inversi
Model selesai dilatih! Sekarang kita uji dengan ujian sesungguhnya menggunakan Data Test.


In [ ]:
# Memprediksi Data Test
y_pred_scaled = model.predict(X_test_seq)

# INVERSI: Mengembalikan skala 0-1 menjadi ug/m3 asli
y_true_inv = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
y_pred_inv = scaler_y.inverse_transform(y_pred_scaled).flatten()

# Hitung Metrik Akademis
rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
mae = mean_absolute_error(y_true_inv, y_pred_inv)
r2 = r2_score(y_true_inv, y_pred_inv)

print(f"=== KINERJA FINAL: DEEP LEARNING LSTM ===")
print(f"  RMSE : {rmse:.2f} ug/m3")
print(f"  MAE  : {mae:.2f} ug/m3")
print(f"  R^2  : {r2:.4f}")


## 7. Visualisasi Prediksi LSTM vs Aktual


In [ ]:
# Ambil 14 Hari pertama (14 * 24 jam = 336 baris)
subset_len = 336

# Hati-hati dengan index waktu. Karena kita pakai sliding window 24 jam,
# Prediksi pertama pada Data Test adalah milik jam ke-25.
time_axis = test_scaled.index[TIME_STEPS : TIME_STEPS + subset_len]

actual = y_true_inv[:subset_len]
lstm_pred = y_pred_inv[:subset_len]

plt.figure(figsize=(16, 6))
plt.plot(time_axis, actual, label='Aktual (PM2.5)', color='black', linewidth=2)
plt.plot(time_axis, lstm_pred, label='Prediksi LSTM', color='magenta', alpha=0.9, linestyle='--')

plt.title("Perbandingan Prediksi LSTM vs Aktual (2 Minggu Pertama Set Test)")
plt.ylabel("PM2.5 Concentration (ug/m3)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Menyimpan Model LSTM
Menyimpan "otak" LSTM ke dalam format standard Keras. Model ini akan kita panggil di **Fase 5** nanti untuk membongkar alasannya (menggunakan XAI/SHAP).


In [ ]:
MODEL_DIR = r'D:\Kuliah Praktik\KP BRIN\models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Format .keras adalah standar terbaru yang direkomendasikan TensorFlow
model_path = os.path.join(MODEL_DIR, 'lstm_model.keras')
model.save(model_path)

print(f"[OK] Otak Jaringan Saraf LSTM berhasil disimpan secara permanen di:")
print(model_path)
